In [1]:
!pip -q install openai datasets pandas tqdm tenacity scikit-learn

In [2]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
print("API key loaded securely.")

Enter your OpenAI API key: ··········
API key loaded securely.


In [3]:
import os
import json
import time
import pandas as pd

from json import JSONDecodeError
from tqdm import tqdm
from collections import Counter
from datasets import load_dataset
from openai import OpenAI, RateLimitError
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

MODEL = "gpt-5.5"
DATASET_NAME = "ChanceFocus/flare-fnxl"

# First test with 5 or 20.
# Then set LIMIT = None for the full test set.
LIMIT = 100

dataset = load_dataset(DATASET_NAME)

split_name = "test" if "test" in dataset else list(dataset.keys())[0]
split_data = dataset[split_name]

print(dataset)
print("Split:", split_name)
print("Columns:", split_data.column_names)
print("Example:", split_data[0])


def get_column_name(example, possible_names):
    for name in possible_names:
        if name in example:
            return name

    raise ValueError(
        f"Cannot find any column from {possible_names}. "
        f"Available columns are: {list(example.keys())}"
    )


example = split_data[0]
token_col = get_column_name(example, ["token", "tokens"])
label_col = get_column_name(example, ["label", "labels", "tags"])

print("Token column:", token_col)
print("Label column:", label_col)


def normalize_label(label):
    return str(label).strip()


# Build label set from the dataset.
all_labels = set()

for row in split_data:
    for label in row[label_col]:
        all_labels.add(normalize_label(label))

LABELS = ["O"] + sorted([x for x in all_labels if x != "O"])

print("Number of labels:", len(LABELS))
print("First 20 labels:", LABELS[:20])

# Use short label codes to reduce output cost.
label_to_code = {label: f"L{i}" for i, label in enumerate(LABELS)}
code_to_label = {code: label for label, code in label_to_code.items()}

VALID_CODES = list(code_to_label.keys())

LABEL_CODE_SCHEMA = {
    "type": "object",
    "properties": {
        "labels": {
            "type": "array",
            "items": {
                "type": "string",
                "enum": VALID_CODES
            }
        }
    },
    "required": ["labels"],
    "additionalProperties": False
}


def build_label_mapping_text():
    lines = []
    for label in LABELS:
        code = label_to_code[label]
        lines.append(f"{code}: {label}")
    return "\n".join(lines)


LABEL_MAPPING_TEXT = build_label_mapping_text()


SYSTEM_PROMPT = """
You are a financial numeric extreme labeling model.

Task:
Given a list of tokens from a financial sentence, assign exactly one label code to each token.

The labels describe the semantic role of financial numeric values.
Most non-relevant tokens should be labeled as O.

Rules:
1. Return exactly one label code for each input token.
2. The number of output labels must equal the number of input tokens.
3. Use only the allowed label codes.
4. Do not add explanations.
5. Return only valid JSON following the schema.
"""


@retry(
    retry=retry_if_exception_type((RateLimitError, JSONDecodeError, ValueError)),
    wait=wait_exponential(multiplier=5, min=10, max=120),
    stop=stop_after_attempt(5)
)
def call_openai_fnxl(tokens):
    indexed_tokens = "\n".join([f"{i}: {tok}" for i, tok in enumerate(tokens)])

    response = client.responses.create(
        model=MODEL,
        input=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Allowed label codes:\n{LABEL_MAPPING_TEXT}\n\n"
                    f"Number of tokens: {len(tokens)}\n\n"
                    f"Tokens:\n{indexed_tokens}\n\n"
                    f"Return exactly {len(tokens)} label codes."
                )
            }
        ],
        text={
            "format": {
                "type": "json_schema",
                "name": "fnxl_sequence_labeling",
                "schema": LABEL_CODE_SCHEMA,
                "strict": True
            }
        },
        reasoning={
            "effort": "low"
        },
        max_output_tokens=1200
    )

    raw_output = response.output_text

    if raw_output is None or raw_output.strip() == "":
        raise ValueError("Empty model output")

    try:
        parsed = json.loads(raw_output)
    except JSONDecodeError:
        print("Raw model output:")
        print(repr(raw_output))
        raise

    if "labels" not in parsed:
        raise ValueError(f"Missing labels field in model output: {parsed}")

    return parsed


def fix_prediction_length(pred_codes, target_len):
    cleaned = []

    for code in pred_codes:
        code = str(code).strip()
        if code in code_to_label:
            cleaned.append(code)
        else:
            cleaned.append(label_to_code["O"])

    if len(cleaned) < target_len:
        cleaned = cleaned + [label_to_code["O"]] * (target_len - len(cleaned))

    if len(cleaned) > target_len:
        cleaned = cleaned[:target_len]

    return cleaned


def codes_to_labels(codes):
    return [code_to_label.get(code, "O") for code in codes]


def labels_to_spans(labels):
    spans = []

    for i, label in enumerate(labels):
        if label == "O":
            continue

        spans.append((i, i + 1, label))

    return spans


def compute_metrics(gold_all, pred_all):
    correct_tokens = 0
    total_tokens = 0
    exact_match_count = 0

    tp = 0
    fp = 0
    fn = 0

    for gold_labels, pred_labels in zip(gold_all, pred_all):
        total_tokens += len(gold_labels)
        correct_tokens += sum(g == p for g, p in zip(gold_labels, pred_labels))

        if gold_labels == pred_labels:
            exact_match_count += 1

        gold_spans = Counter(labels_to_spans(gold_labels))
        pred_spans = Counter(labels_to_spans(pred_labels))

        tp += sum((gold_spans & pred_spans).values())
        fp += sum((pred_spans - gold_spans).values())
        fn += sum((gold_spans - pred_spans).values())

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0
    token_accuracy = correct_tokens / total_tokens if total_tokens > 0 else 0.0
    exact_match_accuracy = exact_match_count / len(gold_all) if len(gold_all) > 0 else 0.0

    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "token_accuracy": token_accuracy,
        "exact_match_accuracy": exact_match_accuracy,
        "true_positive": tp,
        "false_positive": fp,
        "false_negative": fn
    }


eval_data = split_data if LIMIT is None else split_data.select(range(LIMIT))

rows = []
gold_all = []
pred_all = []

for idx, row in enumerate(tqdm(eval_data)):
    tokens = row[token_col]
    gold_labels = [normalize_label(x) for x in row[label_col]]

    try:
        result = call_openai_fnxl(tokens)
        pred_codes = result.get("labels", [])
        pred_codes = fix_prediction_length(pred_codes, len(gold_labels))
        pred_labels = codes_to_labels(pred_codes)
        error = ""
    except Exception as e:
        pred_labels = ["O"] * len(gold_labels)
        pred_codes = [label_to_code["O"]] * len(gold_labels)
        error = str(e)
        print(f"Error at row {idx}: {e}")

    gold_all.append(gold_labels)
    pred_all.append(pred_labels)

    rows.append({
        "idx": idx,
        "id": row.get("id", ""),
        "text": row.get("text", ""),
        "tokens": tokens,
        "gold_labels": gold_labels,
        "pred_codes": pred_codes,
        "pred_labels": pred_labels,
        "gold_spans": labels_to_spans(gold_labels),
        "pred_spans": labels_to_spans(pred_labels),
        "exact_match": gold_labels == pred_labels,
        "error": error
    })

    # Slow down to reduce RateLimitError.
    time.sleep(5)


metrics = compute_metrics(gold_all, pred_all)

print("\n===== GPT-5.5 FLARE-FNXL Evaluation =====")
print(f"Dataset: {DATASET_NAME}")
print(f"Split: {split_name}")
print(f"Model: {MODEL}")
print(f"Samples evaluated: {len(eval_data)}")
print(f"Precision / Correctness Rate: {metrics['precision']:.4f}")
print(f"Recall: {metrics['recall']:.4f}")
print(f"F1 Score: {metrics['f1_score']:.4f}")
print(f"Exact Match Accuracy: {metrics['exact_match_accuracy']:.4f}")
print(f"Token Accuracy: {metrics['token_accuracy']:.4f}")
print(f"TP: {metrics['true_positive']}")
print(f"FP: {metrics['false_positive']}")
print(f"FN: {metrics['false_negative']}")

df = pd.DataFrame(rows)
df.to_csv("gpt55_flare_fnxl_evaluation_results.csv", index=False)

with open("gpt55_flare_fnxl_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

label_map = {
    "label_to_code": label_to_code,
    "code_to_label": code_to_label
}

with open("gpt55_flare_fnxl_label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

num_errors = df["error"].astype(str).str.len().gt(0).sum()

print("\n===== Error Summary =====")
print(f"Failed samples: {num_errors}")
print(f"Total samples: {len(df)}")
print(f"Failure rate: {num_errors / len(df):.4f}")

pd.set_option("display.max_colwidth", 120)
df.head()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/523 [00:00<?, ?B/s]

data/test-00000-of-00001-35c7f3863a79cbc(…):   0%|          | 0.00/315k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/318 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['id', 'query', 'answer', 'text', 'label', 'token'],
        num_rows: 318
    })
})
Split: test
Columns: ['id', 'query', 'answer', 'text', 'label', 'token']
Example: {'id': 'fnxl0', 'query': "In the task of Financial Numeric Extreme Labelling (FNXL), your job is to identify and label the semantic role of each token in a sentence. The labels can include O, B-BusinessCombinationContingentConsiderationArrangementsRangeOfOutcomesValueHigh, B-VariableInterestEntityOwnershipPercentage, B-GainLossOnDispositionOfAssets1, B-IndefiniteLivedIntangibleAssetsExcludingGoodwill, B-MarketingAndAdvertisingExpense, B-ReportingUnitPercentageOfFairValueInExcessOfCarryingAmount, B-CapitalizedComputerSoftwareNet, B-BusinessCombinationConsiderationTransferredEquityInterestsIssuedAndIssuable, B-LitigationSettlementExpense, B-DefinedBenefitPlanExpectedAmortizationOfGainLossNextFiscalYear, B-DeferredCompensationArrangementWithIndividualCompensationExpense, B-

100%|██████████| 100/100 [15:50<00:00,  9.50s/it]


===== GPT-5.5 FLARE-FNXL Evaluation =====
Dataset: ChanceFocus/flare-fnxl
Split: test
Model: gpt-5.5
Samples evaluated: 100
Precision / Correctness Rate: 0.3903
Recall: 0.7784
F1 Score: 0.5199
Exact Match Accuracy: 0.3300
Token Accuracy: 0.9382
TP: 137
FP: 214
FN: 39

===== Error Summary =====
Failed samples: 0
Total samples: 100
Failure rate: 0.0000


,idx,id,text,tokens,gold_labels,pred_codes,pred_labels,gold_spans,pred_spans,exact_match,error
0,0,fnxl0,"Compensation expense recognized for all of the Company's deferred compensation plans was $0.6 million, $1.7 million ...","[Compensation, expense, recognized, for, all, of, the, Company, 's, deferred, compensation, plans, was, $, 0.6, mill...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-DeferredCompensationArrangementWithIndividualCompensationExpense, O, O,...","[L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L19, L0, L0, L0, L19, L0, L0, L0, L19, L0, L0, L0, L0, L0, ...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-DeferredCompensationArrangementWithIndividualCompensationExpense, O, O,...","[(14, 15, B-DeferredCompensationArrangementWithIndividualCompensationExpense), (18, 19, B-DeferredCompensationArrang...","[(14, 15, B-DeferredCompensationArrangementWithIndividualCompensationExpense), (18, 19, B-DeferredCompensationArrang...",True,
1,1,fnxl1,The $10.9 million fair value of the contingent consideration element as of the acquisition date was estimated by app...,"[The, $, 10.9, million, fair, value, of, the, contingent, consideration, element, as, of, the, acquisition, date, wa...","[O, O, B-BusinessCombinationContingentConsiderationLiability, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, ...","[L0, L6, L6, L6, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0,...","[O, B-BusinessCombinationContingentConsiderationLiability, B-BusinessCombinationContingentConsiderationLiability, B-...","[(2, 3, B-BusinessCombinationContingentConsiderationLiability)]","[(1, 2, B-BusinessCombinationContingentConsiderationLiability), (2, 3, B-BusinessCombinationContingentConsiderationL...",False,
2,2,fnxl2,"During 2019, we recorded additions to our ROU assets associated with operating leases of $88.5 million.","[During, 2019, ,, we, recorded, additions, to, our, ROU, assets, associated, with, operating, leases, of, $, 88.5, m...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-RightOfUseAssetObtainedInExchangeForOperatingLeaseLiability, O, O]","[L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L86, L86, L86, L0]","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-RightOfUseAssetObtainedInExchangeForOperatingLeaseLiability, B-Right...","[(16, 17, B-RightOfUseAssetObtainedInExchangeForOperatingLeaseLiability)]","[(15, 16, B-RightOfUseAssetObtainedInExchangeForOperatingLeaseLiability), (16, 17, B-RightOfUseAssetObtainedInExchan...",False,
3,3,fnxl3,Equity in earnings of certain of our joint ventures includes the amortization of the Company’s excess purchase price...,"[Equity, in, earnings, of, certain, of, our, joint, ventures, includes, the, amortization, of, the, Company, ’s, exc...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-EquityMethodInvestmentDifferenceBetweenCarryingAmo...","[L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L35, L0, L0, L0, L0, L0, L0, L0...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-EquityMethodInvestmentDifferenceBetweenCarryingAmo...","[(21, 22, B-EquityMethodInvestmentDifferenceBetweenCarryingAmountAndUnderlyingEquity)]","[(21, 22, B-EquityMethodInvestmentDifferenceBetweenCarryingAmountAndUnderlyingEquity)]",True,
4,4,fnxl4,"Included in other assets are deferred financing costs (net of accumulated amortization), related to the revolver, of...","[Included, in, other, assets, are, deferred, financing, costs, (, net, of, accumulated, amortization, ), ,, related,...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-DeferredFinanceCostsNet, O, O, O, B-DeferredFin...","[L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L0, L21, L21, L21, L0, L21, L21, L2...","[O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, O, B-DeferredFinanceCostsNet, B-DeferredFinanceCostsNet...","[(22, 23, B-DeferredFinanceCosts